# Phase 4 — Feature Extraction

**Outputs:**
- `corpus_embeddings.npy` — all passages encoded with `all-MiniLM-L6-v2` (kNN backbone)
- `labeled_embeddings.npy` — training set encoded
- `features.parquet` — hedge density, scope flag, proprietary standard flag, quantitative target flag, Flesch-Kincaid, beauty unregulated terms

**Carbon tracked:** embedding block (label: `embedding_corpus`)

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import textstat
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from codecarbon import EmissionsTracker

In [2]:
ROOT        = Path('..').resolve()
RAW_DIR     = ROOT / 'data' / 'raw'
PASSAGES    = RAW_DIR / 'extracted' / 'passages.jsonl'
LABELED     = RAW_DIR / 'labeled' / 'labeled_passages.jsonl'
FEAT_DIR    = RAW_DIR / 'features'
FEAT_DIR.mkdir(parents=True, exist_ok=True)
CARBON_DIR  = ROOT / 'results' / 'carbon'
CARBON_DIR.mkdir(parents=True, exist_ok=True)

with open(PASSAGES) as f:
    corpus = [json.loads(l) for l in f]
with open(LABELED) as f:
    labeled = [json.loads(l) for l in f]

corpus_texts  = [p['text'] for p in corpus]
labeled_texts = [p['text'] for p in labeled]
print(f"Corpus: {len(corpus_texts):,} passages | Labeled train: {len(labeled_texts)}")

Corpus: 1,279 passages | Labeled train: 52


## Embeddings — `all-MiniLM-L6-v2`

Encode all corpus passages and the labeled training set. L2-normalize for cosine similarity.  
**Carbon tracked** under label `embedding_corpus`.

In [3]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

tracker = EmissionsTracker(
    project_name="green-claims-nlp",
    output_dir=str(CARBON_DIR),
    output_file="emissions.csv",
    log_level="error",
)
tracker.start()

corpus_embeddings  = model.encode(corpus_texts,  batch_size=64, show_progress_bar=True, convert_to_numpy=True)
labeled_embeddings = model.encode(labeled_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

emissions = tracker.stop()
print(f"\nembedding_corpus emissions: {emissions:.4f} kg CO2e")

# L2-normalize for cosine similarity in kNN
corpus_embeddings  = normalize(corpus_embeddings,  norm='l2')
labeled_embeddings = normalize(labeled_embeddings, norm='l2')

np.save(FEAT_DIR / 'corpus_embeddings.npy',  corpus_embeddings)
np.save(FEAT_DIR / 'labeled_embeddings.npy', labeled_embeddings)
print(f"corpus_embeddings:  {corpus_embeddings.shape}")
print(f"labeled_embeddings: {labeled_embeddings.shape}")

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


embedding_corpus emissions: 0.0000 kg CO2e
corpus_embeddings:  (1279, 384)
labeled_embeddings: (52, 384)


## Supplementary features

These are **not inputs to kNN** — they're used in Phase 6 analysis to interpret and explain model outputs.

In [4]:
# ── Hedge word lexicon ────────────────────────────────────────────────────────
# Base: Loughran & McDonald (2011) uncertainty wordlist (selected)
# Domain additions from pilot reading of H&M and Inditex reports
HEDGE_TERMS = {
    # Loughran-McDonald uncertainty
    'approximately', 'uncertain', 'uncertainties', 'unclear', 'ambiguous',
    'depend', 'depends', 'depending', 'contingent', 'indefinite',
    # Domain additions
    'committed to', 'working towards', 'aims to', 'strives to', 'exploring',
    'ambition is', 'journey towards', 'we believe', 'we strive',
    'in a sustainable way', 'positive impact', 'conscious', 'responsible',
    'hope to', 'intend to', 'plan to', 'seek to', 'aspire',
}

def hedge_density(text: str) -> float:
    text_lower = text.lower()
    hits = sum(1 for term in HEDGE_TERMS if term in text_lower)
    words = len(text_lower.split())
    return hits / words if words > 0 else 0.0


# ── Third-party standard anchors (exempt from proprietary flag) ───────────────
THIRD_PARTY_STANDARDS = {
    'sbti', 'science based targets', 'gri', 'zdhc', 'fsc', 'bluesign',
    'oeko-tex', 'oeko tex', 'fair trade', 'fairtrade', 'b corp', 'bcorp',
    'deloitte', 'pwc', 'kpmg', 'ey assurance', 'isae', 'cdp',
    'better cotton initiative', 'rainforest alliance', 'rspo',
}

# ── Proprietary standard terms ────────────────────────────────────────────────
PROPRIETARY_TERMS = {
    'lower-impact', 'lower impact', 'sustainably sourced', 'responsibly made',
    'conscious choice', 'eco-designed', 'eco designed',
    'better cotton',   # flag unless BCI-verified outcome metrics present
}

def proprietary_standard_flag(text: str) -> int:
    text_lower = text.lower()
    has_proprietary = any(t in text_lower for t in PROPRIETARY_TERMS)
    has_third_party = any(t in text_lower for t in THIRD_PARTY_STANDARDS)
    # Flag only if proprietary term present WITHOUT a third-party standard
    return int(has_proprietary and not has_third_party)


# ── Quantitative target regex ─────────────────────────────────────────────────
_QUANT_RE = re.compile(
    r'(\d+\.?\d*\s*%'           # percentage
    r'|\d{4}'                    # year
    r'|\d+\.?\d*\s*(tonnes?|tco2e?|kwh|mwh|gwh|kg|mt))',
    re.IGNORECASE,
)

def has_quantitative_target(text: str) -> int:
    return int(bool(_QUANT_RE.search(text)))


# ── Scope flag ────────────────────────────────────────────────────────────────
OWN_OPS_KW   = {'our stores', 'our offices', 'scope 1', 'scope 2', 'own operations',
                 'our facilities', 'our headquarters', 'our buildings', 'direct operations'}
SUPPLY_KW    = {'supply chain', 'suppliers', 'scope 3', 'value chain',
                 'tier 1', 'tier 2', 'upstream', 'downstream', 'manufacturers',
                 'sourcing', 'raw materials', 'vendor'}

def scope_flag(text: str) -> str:
    text_lower = text.lower()
    own = any(k in text_lower for k in OWN_OPS_KW)
    sup = any(k in text_lower for k in SUPPLY_KW)
    if own and sup: return 'mixed'
    if own:         return 'own_ops'
    if sup:         return 'supply_chain'
    return 'unclear'


# ── Unregulated beauty terms (beauty industry only) ───────────────────────────
BEAUTY_TERMS = {
    'clean', 'natural', 'non-toxic', 'nontoxic', 'reef-safe', 'reef safe',
    'green', 'pure', 'eco', 'gentle', 'plant-based', 'plant based',
}

def beauty_term_flag(text: str, industry: str) -> int:
    if industry != 'clean_beauty':
        return 0
    text_lower = text.lower()
    return int(any(t in text_lower for t in BEAUTY_TERMS))


print("Feature functions defined.")

Feature functions defined.


In [5]:
# Compute all features for the full corpus
rows = []
for p in corpus:
    rows.append({
        'passage_id':          p['passage_id'],
        'brand':               p['brand'],
        'industry':            p['industry'],
        'role':                p['role'],
        'hedge_density':       hedge_density(p['text']),
        'proprietary_flag':    proprietary_standard_flag(p['text']),
        'has_quant_target':    has_quantitative_target(p['text']),
        'scope':               scope_flag(p['text']),
        'flesch_kincaid_grade':textstat.flesch_kincaid_grade(p['text']),
        'beauty_term_flag':    beauty_term_flag(p['text'], p['industry']),
        'text_length_chars':   len(p['text']),
    })

features_df = pd.DataFrame(rows)
features_df.to_parquet(FEAT_DIR / 'features.parquet', index=False)
print(f"Saved {len(features_df):,} rows → {FEAT_DIR / 'features.parquet'}")
features_df.describe()

Saved 1,279 rows → /Users/mandy.sun/green-claims-nlp/data/raw/features/features.parquet


,passage_id,hedge_density,proprietary_flag,has_quant_target,flesch_kincaid_grade,beauty_term_flag,text_length_chars
count,1279.000000,1279.000000,1279.000000,1279.000000,1279.000000,1279.000000,1279.000000
mean,639.000000,0.002615,0.068022,0.792807,16.376718,0.098514,1800.333855
std,369.359807,0.003459,0.251882,0.405454,5.915093,0.298126,1086.208496
min,0.000000,0.000000,0.000000,0.000000,6.019649,0.000000,228.000000
25%,319.500000,0.000000,0.000000,1.000000,13.698154,0.000000,1361.000000
50%,639.000000,0.000000,0.000000,1.000000,15.778186,0.000000,1646.000000
75%,958.500000,0.004386,0.000000,1.000000,17.958065,0.000000,1991.000000
max,1278.000000,0.028986,1.000000,1.000000,105.127321,1.000000,17873.000000
